# Paper figure generation
Run the cells in order. All implementations come from `code/scripts/` and `code/src/`; generated figures are saved to `code/figure-reproduction/generated/notebook/`, separate from the supplied paper assets.
Figure 2 retains the Z-only wavepackets, labels (a)-(f), and dashed I/II/III region boundaries. Figure 4 retains labels (a) and (b).
The command-line equivalent for the current manuscript is `.venv/bin/python code/figure-reproduction/reproduce_figures.py`.

Saved finite-p inputs are read from `data/numerics/`. See [data/README.md](../../data/README.md) for the selected snapshots, parameters, checksums, and regeneration command. Other figures compute their numerical correlators directly.


In [ ]:
from __future__ import annotations

import importlib
import sys
from dataclasses import asdict
from pathlib import Path

from IPython.display import Image, Markdown, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "code" / "scripts" / "generate_paper_figures.py").exists():
            return path
    raise RuntimeError("Could not find repo root from the current working directory")


REPO_ROOT = find_repo_root(Path.cwd())
CODE_ROOT = REPO_ROOT / "code"
CODE_SRC_DIR = CODE_ROOT / "src"

for path in (CODE_ROOT, CODE_SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from large_p import evap_syk as _evap_syk
from scripts import generate_paper_figures as gpf

_evap_syk = importlib.reload(_evap_syk)
gpf = importlib.reload(gpf)

assert Path(gpf.__file__).resolve() == (
    CODE_ROOT / "scripts" / "generate_paper_figures.py"
)

# Generated outputs are separate from the supplied paper assets.
OUT_DIR = REPO_ROOT / "code" / "figure-reproduction" / "generated" / "notebook"

def resolve_saved_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path


def iter_saved_images(obj, prefix: str = ""):
    if isinstance(obj, (str, Path)):
        path = resolve_saved_path(obj)
        if path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            yield prefix.rstrip("."), path
    elif isinstance(obj, dict):
        for key, value in obj.items():
            yield from iter_saved_images(value, f"{prefix}{key}.")


def show_saved(obj):
    shown_any = False
    for label, path in iter_saved_images(obj):
        shown_any = True
        display(Markdown(f"**{label}**  \\n`{gpf.display_path(path)}`"))
        display(Image(filename=str(path)))
    if not shown_any:
        display(Markdown("No saved images found to display."))


def get_large_p_reconstruction(force: bool = False):
    global _large_p_recon, _large_p_recon_params
    snapshot = asdict(params)
    if force or "_large_p_recon" not in globals() or _large_p_recon_params != snapshot:
        _large_p_recon = gpf.build_large_p_reconstruction(params)
        _large_p_recon_params = snapshot
    return _large_p_recon


display(Markdown(f"Repository root: `{REPO_ROOT}`"))

## Parameters
These independent presets match the Methods figure-parameter subsection. `params` controls Figure 2; `size_params` controls Figures 3 and 5; `mi_params` controls the saved-data Figure 4.
The analytic code setting `formation_beta=10` corresponds to physical beta_eta=20 at J=0.5; Figure 4 uses beta=5.7355465869 from its saved payload.


In [ ]:
presets = gpf.manuscript_parameters()
params = presets["geometry"]
size_params = presets["operator_size"]
mi_params = presets["finite_p"]
display(Markdown("Figure presets loaded from the canonical script."))


## Manuscript Figure 2: bulk geometry, panels (a)–(f)
Regenerate panel (a) with dashed region boundaries and labels I, II, III, regenerate the Z-only wavepacket panels, and assemble the two rows in caption order.
The wavepacket panels show only the Z component; the Penrose diagrams use the existing source PNGs.
The full command-line workflow is documented in [the figure map](../figure-reproduction/README.md).
The legacy Fig. 2 heatmap below is now in Methods; legacy Fig. 3 filenames are retained.

In [ ]:
figure2_outputs = gpf.generate_figure2(params, OUT_DIR)
show_saved(figure2_outputs["files"])

## Figure 5 (Methods): two-point function heat map

In [ ]:
size_recon = gpf.build_large_p_reconstruction(size_params)
fig2_path = gpf.plot_two_point_heatmap(size_recon, size_params, OUT_DIR)
show_saved({"fig2": fig2_path})

## Figure 3: bulk operator size

In [ ]:
fig4_outputs = gpf.plot_bulk_operator_sizes(size_recon, size_params, OUT_DIR)

display(
    Markdown(
        f"Requested fixed-v time: `{size_params.size_fixed_v_time}`; "
        f"actual grid index `{fig4_outputs['fixed_v_index']}`, "
        f"time `{fig4_outputs['fixed_v_time']}`"
    )
)
show_saved(fig4_outputs["files"])

## Figure 4(a): Renyi-2 mutual information

This cell uses `mi_params`, not `params`. If no saved two-replica run matches `mi_params`, the cell reports that and does not raise a notebook traceback.

In [ ]:
try:
    mi_data = gpf.compute_renyi2_mi_heatmaps(mi_params)
    fig5_cut_outputs = gpf.plot_renyi2_fixed_v_line_cuts(mi_data, mi_params, OUT_DIR)
    fig5_paths = {
        "heatmap_quantum": gpf.plot_renyi2_heatmap(
            mi_data,
            OUT_DIR,
            "fig5a_renyi2_quantum_mi_heatmap.png",
            data_key="Iq",
            vmax=2.0 * gpf.np.log(2.0),
            label=r"$I^{(2)}_{\rm q}$",
            title=r"Second Renyi quantum mutual information",
            params=mi_params,
        ),
        "heatmap_classical": gpf.plot_renyi2_heatmap(
            mi_data,
            OUT_DIR,
            "fig5a_renyi2_classical_mi_heatmap.png",
            data_key="Icl",
            vmax=gpf.np.log(2.0),
            label=r"$I^{(2)}_{\rm cl}$",
            title=r"Second Renyi classical mutual information",
            params=mi_params,
        ),
        "fixed_v_cuts": fig5_cut_outputs["files"],
    }
    display(Markdown(f"Loaded saved two-replica payload: `{gpf.display_path(mi_data['payload_path'])}`"))
    display(Markdown(f"Fixed-v cut times: `{fig5_cut_outputs['fixed_v_times']}`"))
    show_saved(fig5_paths)
except Exception as exc:
    display(
        Markdown(
            "Fig. 4(a) was not generated. "
            f"Check that `mi_params` matches a saved two-replica run. Details: `{exc}`"
        )
    )

## Figure 4(b): Renyi-2 entropy from action difference

This cell uses the saved two-replica saddles and computes the action difference

\begin{equation}
\Delta i_\eta(t_f)
=
i_{\eta,\chi{\rm -tw}}(t_f)
-
i_{\eta,{\rm disc}}(t_f).
\end{equation}

The plotted entropy density is

\begin{equation}
\frac{S_2^\eta(t_f)}{N_\eta}
=
\operatorname{Re}\Delta i_\eta(t_f).
\end{equation}

The expected bound is

\begin{equation}
0
\leq
\operatorname{Re}\Delta i_\eta
\leq
\log 2.
\end{equation}

In [ ]:
try:
    action_data = gpf.compute_renyi2_action_curve(mi_params)
    fig5b_path = gpf.plot_renyi2_action_difference(action_data, mi_params, OUT_DIR)

    show_saved({"fig5b_action_difference": fig5b_path})

except Exception as exc:
    display(
        Markdown(
            "Fig. 4(b) was not generated. "
            f"Check that `mi_params` matches a saved two-replica run. Details: `{exc}`"
        )
    )

## Manuscript Figure 4: finite-p results, panels (a) and (b)
Assemble the saved mutual-information heatmap (a) and entropy plot (b), with labels matching Figure 2. Run the two preceding figure cells first to refresh the numerical panels, or use their existing PNGs.
The composite assembly is also run by the figure-reproduction wrapper.

In [ ]:
figure4_paths = gpf.generate_figure4(OUT_DIR)
show_saved(figure4_paths)